# 13 — Direct collocation

**Objectives.** Construct Radau collocation coefficients, enforce dynamics at polynomial nodes, and compare the resulting structure with shooting. On each interval, a degree-$d$ polynomial approximates the state. Its derivative equals the ODE at collocation nodes; an interpolation equation connects interval endpoints. This gives high accuracy without numerically integrating inside every NLP evaluation.


In [ ]:
import casadi as ca
import numpy as np
degree=3; tau=[0.0]+ca.collocation_points(degree,'radau')
C=np.zeros((degree+1,degree+1)); D=np.zeros(degree+1); B=np.zeros(degree+1)
for j in range(degree+1):
    polynomial=np.poly1d([1.0])
    for r in range(degree+1):
        if r != j: polynomial *= np.poly1d([1.0,-tau[r]])/(tau[j]-tau[r])
    D[j]=polynomial(1.0)
    derivative=np.polyder(polynomial)
    for r in range(degree+1): C[j,r]=derivative(tau[r])
    B[j]=np.polyint(polynomial)(1.0)
print('nodes',tau,'quadrature weights',B,'sum',B.sum())
np.testing.assert_allclose(B.sum(),1.0)


In [ ]:
N,h=10,0.1; opti=ca.Opti(); X=[opti.variable()]; U=[]; objective=0
opti.subject_to(X[0]==1.0)
for k in range(N):
    uk=opti.variable(); U.append(uk); xc=[X[-1]]+[opti.variable() for _ in range(degree)]
    for j in range(1,degree+1):
        polynomial_derivative=sum(C[r,j]*xc[r] for r in range(degree+1))
        opti.subject_to(polynomial_derivative == h*(-xc[j]+uk))
        objective += h*B[j]*(xc[j]**2 + 0.1*uk**2)
    x_end=sum(D[r]*xc[r] for r in range(degree+1))
    X.append(opti.variable()); opti.subject_to(X[-1]==x_end)
opti.minimize(objective+X[-1]**2); opti.solver('ipopt',{'print_time':False},{'print_level':0})
sol=opti.solve(); print('terminal state',sol.value(X[-1]),'cost',sol.value(objective))
assert abs(sol.value(X[-1])) < 0.5


Path constraints should be enforced at enough points to control between-node violations. For free final time, normalize time to $[0,1]$ and multiply the dynamics/cost by the duration decision.

**Common mistakes:** transposing `C`, forgetting the interval length in dynamics/quadrature, and comparing meshes without checking constraint violation between nodes. **Recap:** collocation turns differential equations into sparse algebraic residuals.
